# Scriptorium — Barton HTR Training (Kaggle, name-only)

Trains a Kraken HTR model to read just the NAME field from each row of the 1850 U.S. Census (Barton, Tioga Co., NY).

Change vs. the first run: labels are now plain names (e.g. `Chls. Howard`) rather than the tagged multi-field format (`N=Chls. Howard | A=34 | ...`). This removes the format-skeleton attractor that dominated the previous training. Also uses longer early-stopping patience and cosine LR schedule.

**Setup:** sidebar → Accelerator → **GPU T4 x2**. Then **Save Version → Save & Run All (Commit)**. Trained model appears in the Output tab.

In [ ]:
# 1. Install kraken (latest; git-lfs is preinstalled on Kaggle).
!pip install -q kraken
!kraken --version
!git lfs --version

In [ ]:
# 2. Clone the repo (LFS pointer for the tarball, then explicit pull).
%cd /kaggle/working
!rm -rf scriptorium-rails
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/kraftinator/scriptorium-rails.git
%cd scriptorium-rails
!git lfs pull -I data/barton_htr_data.tar.gz
!ls -lh data/barton_htr_data.tar.gz

In [ ]:
# 3. Extract crops, rebuild manifest with NAME-ONLY labels, emit .gt.txt.
%cd /kaggle/working/scriptorium-rails
!mkdir -p data-extract
!tar xzf data/barton_htr_data.tar.gz -C data-extract
# Rebuild the manifest with plain-name labels (no tags) using the ground-truth
# JSON that's committed to the repo.
!python python/build_dataset_names.py \
    data/barton_tioga_ny_1850_census.json \
    data-extract/crops \
    data-extract/manifest_names.jsonl
# Emit companion .gt.txt files next to each crop (kraken's `-f path` format).
!python python/prep_kraken_gt.py data-extract/manifest_names.jsonl
# Build the training image list.
!find data-extract/crops -name 'line_*.png' | while read p; do [ -f "${p%.png}.gt.txt" ] && echo "$p"; done > data-extract/train_images.lst
!wc -l data-extract/train_images.lst

In [ ]:
# 4. Verify GPU.
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 5. Train. Batch 32 on GPU, longer early-stopping patience (15), cosine LR.
!mkdir -p /kaggle/working/models
!ketos -v -d cuda:0 --workers 4 train \
    -B 32 \
    -o /kaggle/working/models/barton_htr \
    -q early \
    --lag 15 \
    --schedule cosine \
    -F 1.0 \
    -f path \
    -t data-extract/train_images.lst

In [ ]:
# 6. Locate whatever checkpoint kraken wrote (kraken 7.x writes .ckpt files
#    into `-o`'s directory, then converts the best to a safetensors alongside).
#    Copy the safetensors (or fall back to the newest .ckpt) to /kaggle/working
#    so it shows up in the Output tab.
!echo "--- models tree ---"
!find /kaggle/working/models -type f 2>/dev/null | head -20
!echo "--- picking best output ---"
!best=$(ls -t /kaggle/working/models/barton_htr/best_*.safetensors 2>/dev/null | head -1) && \
  echo "safetensors: $best" && \
  cp "$best" /kaggle/working/barton_htr.safetensors && \
  ls -lh /kaggle/working/barton_htr.safetensors || echo "no safetensors"
!latest_ckpt=$(ls -t /kaggle/working/models/barton_htr/*.ckpt 2>/dev/null | head -1) && \
  echo "ckpt: $latest_ckpt" && \
  cp "$latest_ckpt" /kaggle/working/barton_htr.ckpt && \
  ls -lh /kaggle/working/barton_htr.ckpt || echo "no ckpt"